# Kiểm thử End-to-End: Sinh Text bằng LLaVA và Dự đoán bằng MulCo
Mô phỏng pipeline hệ thống thực tế: 
1. Nhận ảnh gốc đầu vào (không có text đi kèm).
2. Gọi mô hình LLaVA để sinh văn bản mô tả đặc điểm, triệu chứng trên lá.
3. Đưa ảnh và văn bản vừa sinh vào mô hình MulCo để đưa ra dự đoán phân loại bệnh cuối cùng.

In [1]:
import os
import sys
import time
from pathlib import Path
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel, AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

# Tự động tìm thư mục gốc (chứa src)
current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))
print(f"Project Root: {PROJECT_ROOT}")

from src.datasets.multimodal_raw_dataset import MultiModalRawDataset
from src.models.backbones.vision.convnext_cbam import ConvNeXt_CBAM
from src.models.fusion.mulco_fusion import MulCoFusionBlock
from src.models.multimodal.mulco_classifier import Conv1x1Classifier

Project Root: /media/data3/users/luongdth/MulCo-PlantNet


## 1. Cấu hình & Định nghĩa Model MulCo

In [2]:
class MulCoEndToEnd(nn.Module):
    def __init__(self, num_classes=28, proj_dim=512):
        super().__init__()
        self.image_backbone = ConvNeXt_CBAM(num_classes=num_classes)
        self.text_backbone = AutoModel.from_pretrained("roberta-base")
        
        self.img_proj = nn.Conv2d(1024, proj_dim, kernel_size=1)
        self.txt_proj = nn.Linear(768, proj_dim)
        
        self.fusion_blocks = nn.ModuleList([
            # Sử dụng 1 khối Fusion duy nhất
            MulCoFusionBlock(dim=proj_dim, num_heads=8) for _ in range(1)
        ])
        
        self.classifier = Conv1x1Classifier(in_channels=proj_dim, num_classes=num_classes)

    def forward(self, images, input_ids, attention_mask):
        img_feat = self.image_backbone.forward_features_spatial(images) 
        txt_out = self.text_backbone(input_ids=input_ids, attention_mask=attention_mask)
        txt_feat = txt_out.last_hidden_state
        
        img_feat = self.img_proj(img_feat)
        txt_feat = self.txt_proj(txt_feat)
        
        for block in self.fusion_blocks:
            img_feat, txt_feat = block(img_feat, txt_feat)
            
        return self.classifier(img_feat)

## 2. Load Models (LLaVA & MulCo)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Khởi tạo LLaVA
print("Loading LLaVA-1.5-7B...")
llava_processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
llava_model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=quantization_config,
    device_map="auto"
)

# Khởi tạo MulCo
print("Loading MulCo...")
mulco_model = MulCoEndToEnd(num_classes=28).to(device)
ckpt_path = os.path.join(PROJECT_ROOT, "archive", "mulco_depth_aug", "best_fine_tuned_model.pth")
mulco_model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
mulco_model.eval()

# Bộ mã hóa văn bản của RoBERTa và Phép biến đổi ảnh
text_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
mulco_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Using device: cuda
Loading LLaVA-1.5-7B...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Loading MulCo...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 3. Khởi tạo Pipeline Dự Đoán End-to-End

In [4]:
def predict_end_to_end(image_path):
    raw_image = Image.open(image_path).convert("RGB")
    
    # --- Bước 1: Sinh Text từ LLaVA ---
    prompt_text = (
        "Act as an expert plant pathologist conducting a detailed visual inspection. Provide a comprehensive, structured description of the leaf's condition, focusing exclusively on pathological and physiological features.\n\n"
        "CRITICAL RULES:\n"
        "- DO NOT state, guess, or imply the name of the plant species (e.g., do not use words like Tomato, Apple, Corn, Potato).\n"
        "- DO NOT name the specific disease or pathogen (e.g., do not say Early Blight, Rust, Mosaic Virus, Scab).\n"
        "- DO NOT provide a final diagnosis.\n"
        "- Restrict your output STRICTLY to observable visual symptoms. Do not describe the background, lighting, or irrelevant objects.\n\n"
        "If the leaf appears completely healthy:\n"
        "Describe its healthy state in detail. Note the uniform coloration, intact structural integrity, natural texture, and the explicit absence of any lesions, discoloration, pest damage, fungal growth, or viral deformations.\n\n"
        "If the leaf exhibits signs of disease, pathogens, or pest damage, systematically describe the symptoms using the following aspects:\n"
        "1. Color & Pigmentation: Describe any abnormal discoloration, including general chlorosis (yellowing), distinct mosaic/mottling patterns, or specific color changes in affected areas.\n"
        "2. Structural Deformation: Note any physical distortions such as leaf curling, wrinkling, stunting, or wilting.\n"
        "3. Spot & Lesion Morphology: Detail the characteristics of any spots or lesions—specify their color, shape (e.g., angular, circular, irregular), internal patterns (e.g., target-like concentric rings, water-soaked appearance), and whether they have distinct borders or chlorotic halos.\n"
        "4. Pathogen & Pest Signs: Report any visible evidence of the causal agent, such as fungal fuzzy mold, powdery mildew, rust pustules, or pest indicators like spider mite stippling, webbing, or insect feeding holes.\n"
        "5. Tissue Necrosis & Distribution: Describe the extent of dead tissue (necrosis), blighting, structural collapse, and how these symptoms are distributed across the leaf (e.g., at the margins, interveinal, or randomly scattered).\n\n"
        "Provide the final output as a cohesive, professional pathological report in a single well-connected paragraph. Maximize the use of precise botanical and pathological terminology without violating the critical rules.\n"
    )
    prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
    inputs = llava_processor(text=prompt, images=raw_image, return_tensors="pt").to(device, torch.float16)
    
    with torch.no_grad():
        # Dùng Beam Search thay vì Sample ngẫu nhiên để câu sinh ra ở tập test ổn định và ít ảo giác hơn
        output = llava_model.generate(
            **inputs, 
            max_new_tokens=256,
            num_beams=3,
            do_sample=False
        )
    generated_text = llava_processor.decode(output[0], skip_special_tokens=True)
    
    # Cắt bỏ phần prompt để lấy đúng câu trả lời
    caption = generated_text.split("ASSISTANT:")[-1].strip()
    
    # --- Bước 2: Tiền xử lý cho MulCo ---
    img_tensor = mulco_transform(raw_image).unsqueeze(0).to(device)
    
    text_tokens = text_tokenizer(
        [caption], 
        padding=True, 
        truncation=True, 
        max_length=256, 
        return_tensors="pt"
    )
    input_ids = text_tokens.input_ids.to(device)
    attn_mask = text_tokens.attention_mask.to(device)
    
    # --- Bước 3: Đưa vào MulCo dự đoán ---
    with torch.no_grad():
        logits = mulco_model(img_tensor, input_ids, attn_mask)
        pred = torch.argmax(logits, dim=1).item()
        
    torch.cuda.empty_cache()
    return pred, caption

## 4. Chạy Đánh Giá Thực Tế

In [5]:
class_mapping = {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf': 25, 'grape_leaf': 26, 'grape_leaf_black_rot': 27}
idx_to_class = {v: k for k, v in class_mapping.items()}

# Sử dụng Dataset để lấy danh sách ảnh Test (bỏ qua json vì mình sẽ sinh caption bằng LLaVA)
test_dataset = MultiModalRawDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/processed/PlantDocSplited_depth_AUG/test"),
    caption_root=os.path.join(PROJECT_ROOT, "data/processed/captions_LLaVA_depth_AUG/test"),
    transform=None,  # Để None vì ảnh sẽ được xử lý trong hàm predict_end_to_end
    use_depth_suppressed=False,
    strict_caption_match=False
)
print(f"Total items to test: {len(test_dataset)}")

all_preds = []
all_labels = []

# Mẹo: Ban đầu bạn nên đặt `num_samples_to_test = 10` để chạy thử xem code chạy ổn không.
# Nếu chạy ok, đổi sang `len(test_dataset)` để chạy hết.
num_samples_to_test = len(test_dataset)

for idx in tqdm(range(num_samples_to_test), desc="End-to-End Inference"):
    item = test_dataset[idx]
    image_path = item["image_path"]
    true_label = item["label"]
    
    start_time = time.time()
    pred, generated_caption = predict_end_to_end(image_path)
    latency = time.time() - start_time
    
    print(f"\n--- Image: {Path(image_path).name} | Time: {latency:.2f}s ---")
    print(f"Generated Caption:\n{generated_caption}")
    print(f"-> Predicted class: {idx_to_class[pred]} ({pred}) | True class: {idx_to_class[true_label]} ({true_label})")
    
    all_preds.append(pred)
    all_labels.append(true_label)

acc = accuracy_score(all_labels, all_preds)
print(f"\n====================================")
print(f"End-to-End Accuracy (on {len(all_labels)} samples): {acc:.4f}")
print(f"====================================")


[MultiModalRawDataset] Total selected images: 250
[MultiModalRawDataset] Valid samples: 250
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Matched by external mapping: 0
[MultiModalRawDataset] Num classes: 28
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf'

End-to-End Inference:   0%|          | 0/250 [00:00<?, ?it/s]


--- Image: test_Apple Scab Leaf_1.jpg | Time: 15.88s ---
Generated Caption:
The leaf in the image appears to be affected by a disease or pathogen, as evidenced by the presence of spots, lesions, and discoloration. The spots are scattered across the leaf and vary in size, shape, and color. Some of the spots exhibit a chlorotic halo, while others have distinct borders. The leaf also shows signs of structural deformation, such as wrinkling and stunting. Additionally, there is evidence of pest damage, including insect feeding holes and spider mite stippling. Overall, the leaf displays a combination of pathological and physiological symptoms, indicating a complex issue affecting the plant's health.
-> Predicted class: Apple_Scab_Leaf (0) | True class: Apple_Scab_Leaf (0)

--- Image: test_Apple Scab Leaf_10.jpg | Time: 17.42s ---
Generated Caption:
The leaf in the image exhibits signs of disease and pest damage. There are multiple spots and lesions scattered across the leaf, which appear to